## A real build — a Flask app

Let's put it together and build a tiny Flask app. Three files plus a `.dockerignore`:

```python
# app.py
from flask import Flask
app = Flask(__name__)

@app.get("/")
def root():
    return {"message": "hello from docker"}

if __name__ == "__main__":
    app.run(host="0.0.0.0", port=8080)
```

```
# requirements.txt
flask==3.0.3
```

```dockerfile
FROM python:3.12-slim
ENV PYTHONDONTWRITEBYTECODE=1 PYTHONUNBUFFERED=1
WORKDIR /app

# Deps first — this layer caches across source-code changes
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Source after — this layer invalidates when code changes
COPY app.py .

RUN useradd --create-home --shell /bin/bash appuser
USER appuser
EXPOSE 8080
CMD ["python", "app.py"]
```

```bash
docker build -t flask-hello .
docker run --rm -p 8080:80 flask-hello   # then curl localhost:8080
```

Every line here is something from the last few sections — but the **ordering** is the lesson, and it's the single highest-value habit in Dockerfile authoring. Watch the two `COPY`s:

- `COPY requirements.txt` + `RUN pip install` come **first**. Dependencies change rarely, so on most rebuilds these layers are a **cache hit** — `pip install` doesn't re-run.
- `COPY app.py` comes **after**. Your source changes constantly, but it only busts the cache *from that line down* — deps stay cached.

Flip the order (copy all source, then install) and *every* code edit re-runs `pip install`, turning a one-second rebuild into a minute. **Order instructions least-changing to most-changing**, and the build cache does the rest.